# GLAMMAR AI — Whisper Small

Upload audio (max 25 MB) and transcribe it to text.

1. Sign into Google. Choose **Runtime → Change runtime type → T4 GPU** (free GPUs are not guaranteed).
2. Choose **Runtime → Run all**, approve after reviewing, then use the controls at the bottom.

Weights download into this temporary session. Save results before it disconnects. This notebook runs interactively in Colab — it is not an API server. Free Colab policies: https://research.google.com/colaboratory/faq.html

In [ ]:
%pip -q install transformers accelerate ipywidgets soundfile librosa
print("Dependencies installed. Continue below.")

In [ ]:
import os, tempfile
from pathlib import Path
import ipywidgets as w
from IPython.display import display, Markdown
from google.colab import output, userdata
from transformers import pipeline
import torch
output.enable_custom_widget_manager()

if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
print("GPU:", torch.cuda.get_device_name(0))

try:
    TOKEN = userdata.get("HF_TOKEN")
except Exception:
    TOKEN = None

pipe = pipeline("automatic-speech-recognition", model="openai/whisper-small", token=TOKEN, torch_dtype=torch.float16, device=0)
print("Whisper ready.")

upload = w.FileUpload(accept=".wav,.mp3,.m4a,.flac,.ogg", multiple=False, description="Upload audio")
send = w.Button(description="Transcribe", button_style="primary", icon="play")
log = w.Output()

def on_send(_):
    send.disabled = True
    with log:
        try:
            if not upload.value:
                raise ValueError("Upload an audio file first.")
            values = upload.value
            item = next(iter(values.values())) if isinstance(values, dict) else values[0]
            raw = bytes(item["content"])
            if len(raw) > 25 * 1024 * 1024:
                raise ValueError("Use an audio file smaller than 25 MB.")
            path = None
            try:
                with tempfile.NamedTemporaryFile(delete=False, suffix=Path(item["name"]).suffix) as f:
                    path = f.name
                    f.write(raw)
                text = pipe(path, chunk_length_s=30, batch_size=1)["text"]
                display(Markdown(text))
                Path("/content/glammar-transcript.txt").write_text(text)
                print("Saved in the Colab Files panel: glammar-transcript.txt")
            finally:
                if path and os.path.exists(path):
                    os.remove(path)
        except Exception as e:
            print("Could not complete:", e)
        finally:
            send.disabled = False

send.on_click(on_send)
display(w.VBox([upload, send, log]))